In [31]:
from sqlalchemy import create_engine

# 1. Define your connection details
HOST = ""
PORT = ""
USER = ""
PASSWORD = ""
DB_NAME = ""

# 2. Create the connection engine
# For PostgreSQL: "postgresql://..."
# For MySQL: "mysql+pymysql://..."
db_url = f"postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}"
engine = create_engine(db_url)

In [32]:
import requests
import numpy as np
import pandas as pd

def get_line_hourly_passengers(office_line_id: str, limit: int = None) -> pd.DataFrame:
    """Fetches hourly ticketing/passenger data for a specific line ID."""
    resource_id = "ef42a264-9da2-41ad-9120-822064fb5433"
    url = "https://data.gov.il/api/3/action/datastore_search"
    params = {
        "resource_id": resource_id,
        "q": f'{{"OfficeLineId": "{office_line_id}"}}',
        "limit": limit
    }

    print(f"Fetching hourly passenger data for Makat {office_line_id}...")
    response = requests.get(url, params=params)
    response.raise_for_status()
    records = response.json().get("result", {}).get("records", [])
    return pd.DataFrame(records)

def get_station_passengers(office_line_id: str, limit: int = None) -> pd.DataFrame:
    """Fetches station-level passenger data."""
    resource_id = "3ad014c3-e0a6-4ba0-9b2b-12a29d273512"
    url = "https://data.gov.il/api/3/action/datastore_search"
    params = {
        "resource_id": resource_id,
        "limit": limit
    }

    print(f"Fetching station passenger data...")
    response = requests.get(url, params=params)
    response.raise_for_status()
    records = response.json().get("result", {}).get("records", [])
    return pd.DataFrame(records)

def get_all_stations_geo() -> pd.DataFrame:
    """Fetches the nationwide MOT bus stops master index containing StationId, lat, and lon."""
    resource_id = "e873e6a2-66c1-494f-a677-f5e77348edb0"
    url = "https://data.gov.il/api/3/action/datastore_search"

    all_records = []
    limit = 5000
    offset = 0

    print("Fetching nationwide bus stops master index for coordinate mapping...")
    while True:
        params = {"resource_id": resource_id, "limit": limit, "offset": offset}
        response = requests.get(url, params=params)
        response.raise_for_status()

        records = response.json().get("result", {}).get("records", [])
        if not records:
            break

        all_records.extend(records)
        offset += limit
        if len(records) < limit:
            break

    df_raw = pd.DataFrame(all_records)
    if df_raw.empty:
        return pd.DataFrame(columns=['StationId', 'lat', 'lon'])

    id_col = next((c for c in ['StationId', 'station_id', 'code'] if c in df_raw.columns), None)
    lat_col = next((c for c in ['Lat', 'lat', 'latitude'] if c in df_raw.columns), None)
    lon_col = next((c for c in ['Long', 'lon', 'lng', 'longitude'] if c in df_raw.columns), None)

    return pd.DataFrame({
        "StationId": df_raw[id_col].astype(str).str.strip(),
        "lat": pd.to_numeric(df_raw[lat_col], errors='coerce'),
        "lon": pd.to_numeric(df_raw[lon_col], errors='coerce')
    }).dropna(subset=['StationId', 'lat', 'lon']).drop_duplicates(subset=['StationId'])

def get_planned_route_flat(office_line_id: str) -> pd.DataFrame:
    """Fetches planned route geometry from Open Bus Stride API with flat stop details."""
    base_url = "https://open-bus-stride-api.hasadna.org.il"
    expected_columns = ['stop_sequence', 'stop_code', 'stop_name', 'lat', 'lon']
    empty_df = pd.DataFrame(columns=expected_columns)

    routes_res = requests.get(f"{base_url}/gtfs_routes/list", params={"route_mkt": str(office_line_id), "limit": 1})
    if routes_res.status_code != 200 or not routes_res.json():
        return empty_df
    gtfs_route_id = routes_res.json()[0]['id']

    rides_res = requests.get(f"{base_url}/gtfs_rides/list", params={"gtfs_route_id": gtfs_route_id, "limit": 1})
    if rides_res.status_code != 200 or not rides_res.json():
        return empty_df

    ride = rides_res.json()[0]
    gtfs_ride_id = ride['id']
    ride_start_time_str = ride.get('start_time', '2026-01-01T00:00:00')
    ride_date = ride_start_time_str[:10]

    print(f"Fetching complete route geometry for Ride {gtfs_ride_id}...")
    ride_stops_res = requests.get(f"{base_url}/gtfs_ride_stops/list", params={
        "gtfs_ride_id": gtfs_ride_id,
        "arrival_time_from": f"{ride_date}T00:00:00Z",
        "arrival_time_to": f"{ride_date}T23:59:59Z",
        "get_count": "false"
    })

    if ride_stops_res.status_code != 200:
        return empty_df

    ride_stops = ride_stops_res.json()
    if not ride_stops or not isinstance(ride_stops, list):
        return empty_df

    route_points = []
    valid_stops = sorted([rs for rs in ride_stops if isinstance(rs, dict)], key=lambda x: x.get('stop_sequence', 0))

    for rs in valid_stops:
        route_points.append({
            "stop_sequence": rs.get('stop_sequence'),
            "stop_code": rs.get('gtfs_stop__code'),
            "stop_name": rs.get('gtfs_stop__name'),
            "lat": rs.get('gtfs_stop__lat'),
            "lon": rs.get('gtfs_stop__lon')
        })

    return pd.DataFrame(route_points, columns=expected_columns)

def get_station_passengers_with_coords(office_line_id: str, limit: int = None) -> pd.DataFrame:
    """Fetches station passenger metrics and enriches them with official coordinates using StationId bridging."""
    df_station = get_station_passengers(office_line_id, limit=limit)
    if df_station.empty:
        return df_station

    df_geo = get_all_stations_geo()

    df_station['StationId_str'] = df_station['StationId'].astype(str).str.strip()
    df_geo['StationId_str'] = df_geo['StationId'].astype(str).str.strip()

    df_merged = pd.merge(
        df_station,
        df_geo[['StationId_str', 'lat', 'lon']],
        on='StationId_str',
        how='left'
    ).drop(columns=['StationId_str'])

    return df_merged

def get_all_active_office_line_ids(single_fetch=False) -> list:
    """
    Fetches all distinct route_mkt (office_line_id) values from the
    Open Bus Stride API.
    """
    base_url = "https://open-bus-stride-api.hasadna.org.il/gtfs_routes/list"

    unique_route_mkts = set()
    limit = 5000  # API chunk size
    offset = 0

    print("Fetching active route IDs from Stride API...")

    while True:
        params = {
            "limit": limit,
            "offset": offset,
            "get_count": "false" # Speeds up the query
        }

        response = requests.get(base_url, params=params)
        if response.status_code != 200:
            print(f"Failed to fetch data at offset {offset}: {response.status_code}")
            break

        data = response.json()
        if not data:
            break  # No more records

        for route in data:
            route_mkt = route.get('route_mkt')
            if route_mkt:
                unique_route_mkts.add(str(route_mkt))

        offset += limit
        print(f"Processed {offset} records, found {len(unique_route_mkts)} unique IDs so far...")

        if single_fetch:
            break

    print(f"✅ Successfully retrieved {len(unique_route_mkts)} unique office line IDs.")
    return list(unique_route_mkts)

In [33]:
# City Center Coordinates (Lat, Lon)
METRO_CENTERS = {
    'Tel Aviv': (32.0780, 34.7818),   # Rabin Sq / City Center
    'Jerusalem': (31.7820, 35.2170),  # Jaffa St / City Center
    'Haifa': (32.8190, 34.9980)       # Downtown / German Colony
}

def haversine_vectorized(lat1, lon1, lat2, lon2):
    """Calculates distance in kilometers between two coordinates."""
    R = 6371.0  # Earth's radius in km

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def add_metro_score(
    df: pd.DataFrame,
    lat_col: str = 'lat',
    lon_col: str = 'lon',
    decay_scale_km: float = 15.0
) -> pd.DataFrame:
    """
    Enriches a DataFrame with distance to the nearest major city center
    (Tel Aviv, Jerusalem, Haifa) and an exponential metropolitan score (0-100).

    Parameters:
      decay_scale_km: Controls how quickly the score decays with distance.
                      With default=15.0 km:
                      - 0 km -> 100
                      - 5 km -> 71.7
                      - 15 km -> 36.8
                      - 30 km -> 13.5
    """
    df = df.copy()

    # Calculate distance to each of the 3 city centers
    dist_dict = {}
    for city, (c_lat, c_lon) in METRO_CENTERS.items():
        dist_dict[city] = haversine_vectorized(df[lat_col], df[lon_col], c_lat, c_lon)

    dist_df = pd.DataFrame(dist_dict, index=df.index)

    # Find closest city and minimum distance
    df['min_metro_dist_km'] = dist_df.min(axis=1).round(2)
    df['closest_metro'] = dist_df.idxmin(axis=1)

    # Exponential decay score formula: 100 * exp(-distance / scale)
    df['metro_score'] = (100 * np.exp(-df['min_metro_dist_km'] / decay_scale_km)).round(1)

    return df

time_map = {
    '12:00 - 14:59 - שפל יום 2': 12,
    '15:00 - 18:59 - שיא ערב': 15,
    '19:00 - 23:59 - שפל ערב': 19,
    '04:00 - 05:59 - שפל לפנות בוקר': 4,
    '06:00 - 08:59 - שיא בוקר': 6,
    '09:00 - 11:59 - שפל יום 1': 9,
    '24:00 - 27:59 - שפל לילה': 0
    }

In [34]:
def get_hour_passanger_per_line(busline):
  limit = None

  # 1. Fetch hourly passenger statistics for the line
  # df_line_time = get_line_hourly_passengers(busline, limit=limit)

  # 2. Fetch station passenger metrics automatically enriched with geographic coordinates (lat, lon)
  df_station_enriched = get_station_passengers_with_coords(busline, limit=limit)
  df_station_scored = add_metro_score(df_station_enriched, lat_col='lat', lon_col='lon')
  busline_metro_score = max(df_station_scored['metro_score'].iloc[0], df_station_scored['metro_score'].iloc[-1])


  # 3. Fetch planned sequential route geometry and stop coordinates from Open Bus Stride
  df_route = get_planned_route_flat(busline)

  # Inspect the outputs
  # print("--- Line Hourly Passengers Shape:", df_line_time.shape)
  # print("--- Station Passengers with Coords Shape:", df_station_scored.shape)
  # print(df_station_scored[['StationId', 'StationName', 'lat', 'lon', 'metro_score']].head())

  # print("--- Planned Route Shape:", df_route.shape)

  # convert time
  df_station_scored['hour'] = df_station_scored['LowOrPeakDescFull'].map(time_map)

  # avg days of month
  day_cols = [f"day_{i}" for i in range(1, 32)]
  df_station_scored['avg_daily_passengers'] = df_station_scored[day_cols].mean(axis=1)

  # avg all stations
  df_hour_passengers = df_station_scored.groupby('hour').agg({'avg_daily_passengers': np.mean})

  df_hour_passengers = df_hour_passengers.reset_index()
  df_hour_passengers['busline'] = busline
  df_hour_passengers['metro_score'] = busline_metro_score

  return df_hour_passengers

In [35]:
import concurrent.futures
import pandas as pd
from tqdm import tqdm  # Optional: Great for showing a progress bar

def fetch_all_lines_parallel(buslines: list, max_workers: int = 10) -> pd.DataFrame:
    """
    Fetches passenger data for multiple bus lines concurrently.

    Args:
        buslines: List of office_line_ids (Makat) to fetch.
        max_workers: Number of concurrent threads. Don't set this too high (e.g., >20)
                     to avoid being rate-limited or blocked by the data.gov.il API.
    """
    all_results = []

    # Create a thread pool
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all API tasks to the pool
        # (Assuming your function is named get_hour_passanger_per_line)
        future_to_busline = {
            executor.submit(get_hour_passanger_per_line, busline): busline
            for busline in buslines
        }

        # tqdm adds a nice progress bar as tasks complete
        for future in tqdm(concurrent.futures.as_completed(future_to_busline), total=len(buslines), desc="Fetching Lines"):
            busline = future_to_busline[future]
            try:
                # Get the resulting DataFrame from the function
                df = future.result()

                if df is not None and not df.empty:
                    # Optional: guarantee the line ID is in the data in case the API omits it
                    df['requested_busline'] = busline
                    all_results.append(df)

            except Exception as exc:
                print(f"❌ Busline {busline} generated an exception: {exc}")

    # Concatenate all individual DataFrames into one large DataFrame
    if all_results:
        print(f"\n✅ Successfully fetched data for {len(all_results)} lines.")
        return pd.concat(all_results, ignore_index=True)
    else:
        print("\n⚠️ No data returned for any bus lines.")
        return pd.DataFrame()

In [36]:
buslines = get_all_active_office_line_ids(single_fetch=True)

Fetching active route IDs from Stride API...
Processed 5000 records, found 2200 unique IDs so far...
✅ Successfully retrieved 2200 unique office line IDs.


In [42]:
df_bus_hour_passengers = fetch_all_lines_parallel(buslines[:1], max_workers=1)

Fetching station passenger data...


Fetching Lines: 100%|██████████| 1/1 [00:00<00:00,  5.14it/s]

❌ Busline 11502 generated an exception: 403 Client Error: Forbidden for url: https://data.gov.il/api/3/action/datastore_search?resource_id=3ad014c3-e0a6-4ba0-9b2b-12a29d273512

⚠️ No data returned for any bus lines.


In [20]:
def compute_line_time_anomaly_scores(
    df_bus_hour_passengers: pd.DataFrame,
    metro_bin_size: float = 10.0
) -> pd.DataFrame:
    """
    Computes a standardized anomaly Z-score for each (buslien, hour) compared
    to peer bus lines with similar metro_scores during the same hour.
    """
    df = df_bus_hour_passengers.copy()

    # Ensure required columns exist
    required_cols = ['buslien', 'metro_score', 'hour', 'avg_daily_passengers']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing expected column: '{col}'")

    # Clean and convert numeric fields
    df['metro_score'] = pd.to_numeric(df['metro_score'], errors='coerce')
    df['hour'] = pd.to_numeric(df['hour'], errors='coerce').astype(int)
    df['avg_daily_passengers'] = pd.to_numeric(df['avg_daily_passengers'], errors='coerce')

    df = df.dropna(subset=required_cols)

    # 1. Stratify metro scores into peer bins (e.g., 0-10, 10-20, etc.)
    df['metro_bin'] = (df['metro_score'] // metro_bin_size) * metro_bin_size

    # 2. Compute peer statistics (mean & std) grouped by Hour and Metro Bin
    peer_stats = df.groupby(['hour', 'metro_bin'])['avg_daily_passengers'].agg(
        peer_mean='mean',
        peer_std='std',
        peer_count='count'
    ).reset_index()

    # Merge peer stats back into the main DataFrame
    df = pd.merge(df, peer_stats, on=['hour', 'metro_bin'], how='left')

    # 3. Handle zero or NaN standard deviations (e.g., bins with 1 or few lines)
    df['peer_std'] = df['peer_std'].fillna(0.0).replace(0, 1e-5)

    # 4. Compute Anomaly Z-Score
    df['anomaly_z_score'] = (
        (df['avg_daily_passengers'] - df['peer_mean']) / df['peer_std']
    ).round(2)

    # 5. Categorize performance status
    def label_anomaly(z):
        if z <= -1.5:
            return 'Under-performing'
        elif z >= 1.5:
            return 'Over-performing'
        return 'Normal'

    df['usage_anomaly_status'] = df['anomaly_z_score'].apply(label_anomaly)

    return df

In [384]:
anomaly_df = compute_line_time_anomaly_scores(df_bus_hour_passengers)
anomaly_df

In [ ]:
# dataframe:
# uuid: E6B7FEAF-4C21-410F-B2B4-886B9A592526
# output_variable:
# config_str:

import google.colabsqlviz.explore_dataframe as _vizcell
_vizcell.explore_dataframe(df_or_df_name='', uuid='E6B7FEAF-4C21-410F-B2B4-886B9A592526')

In [43]:
# visualize a busline route

route_coordinates = df_station_scored[['lat', 'lon']].values.tolist()
route_coordinates = [coord for coord in route_coordinates if coord[0]==coord[0] and coord[1]==coord[1]]
print(route_coordinates)

# Calculate the center of the route for initial map view
center_lat = df_station_scored['lat'].mean()
center_lon = df_station_scored['lon'].mean()

# Step 3: Create a Folium map object
# The 'location' parameter sets the initial center of the map
# 'zoom_start' sets the initial zoom level
import folium
m = folium.Map(location=[center_lat, center_lon], zoom_start=10)

# Step 4: Add the route as a PolyLine to the map
# 'color' sets the line color, 'weight' sets the line thickness
folium.PolyLine(route_coordinates, color="blue", weight=5, opacity=0.7).add_to(m)

# Step 5: Display the map
# In a Colab notebook, simply having the map object as the last line
# in a cell will render it.
m

NameError: name 'route_coordinates' is not defined